# NDSI over Pyrenees in Leafmap with `jupyter-tiler`

To run this notebook, you'll need the dependencies in this repository's `"play"` dependency group.
For most people, installing the right dependencies might look like:

```bash
uv sync --group dev --group test --group play
```

## First, grab some data and do a calculation on it

We want to see that we can display calculated data from an in-memory dataset.

### Get Sentinel 2 data as an Xarray `DataSet`

#### Define a Pyrenees Bounding Box

In [ ]:
from leafmap.plot import bbox_to_gdf

#bbox = [-72.99321, 41.23109, -72.85227, 41.37502]
#bbox = [1.306, 42.527, 1.551, 42.662]
#bbox = [1.0, 42.5, 1.7, 43.1]
bbox = [-1.7, 42.2, 3.0, 43.4]
bbox_to_gdf(bbox).explore()

too slow and need keys

## CDSE STAC Catalog for S2 (too slow and need keys)

#### Query the CDSE STAC Catalog for S2

In [ ]:
import os
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_ACCESS_KEY_ID"] = 'U0ULPGE9ROF6KQM6DBV3'
os.environ["AWS_SECRET_ACCESS_KEY"] = 'GtTlrVGDcx73cqzfSRHJBw9eVf4faMGWeDBzMpDk'
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"

In [ ]:
config_options = {
                  'AWS_VIRTUAL_HOSTING': 'FALSE',
                  'GDAL_HTTP_UNSAFESSL': 'YES',
                  'AWS_HTTPS': 'YES',
                  'AWS_S3_ENDPOINT': "eodata.dataspace.copernicus.eu"}
import rasterio

In [ ]:
worker_env = {
               'AWS_VIRTUAL_HOSTING': 'FALSE',
               'GDAL_HTTP_UNSAFESSL': 'YES',
                  'AWS_ACCESS_KEY_ID': 'U0ULPGE9ROF6KQM6DBV3',
                  'AWS_SECRET_ACCESS_KEY': 'GtTlrVGDcx73cqzfSRHJBw9eVf4faMGWeDBzMpDk',
                  'AWS_HTTPS': 'YES',
                  'AWS_S3_ENDPOINT': "eodata.dataspace.copernicus.eu"}

In [ ]:
from pystac_client import Client

sentinel2_stac_items = (
    Client.open("https://stac.dataspace.copernicus.eu/v1/")
    .search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime="2025-01-01/2025-02-01",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    .item_collection()
)
sentinel2_stac_items

In [ ]:
import odc.stac

sentinel2_dataset = odc.stac.load(
    sentinel2_stac_items,
    bands=["B04_20m", "B03_20m", "B02_20m", "B11_20m"],
    chunks={'x': 2048, 'y': 2048},
    bbox=bbox,
    resolution=20,
    groupby="solar_day",
)
sentinel2_dataset = sentinel2_dataset.rename_vars(B04_20m="red", B03_20m="green", B02_20m="blue", B11_20m="swir16")
sentinel2_dataset

#### Query the Element84 Earth Search STAC catalog for Sentinel2 data

In [ ]:
from pystac_client import Client

sentinel2_stac_items = (
    Client.open("https://earth-search.aws.element84.com/v1")
    .search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime="2024-01-01/2024-02-01",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    .item_collection()
)
sentinel2_stac_items

#### Load the data into an Xarray `Dataset`

In [ ]:
import odc.stac

sentinel2_dataset = odc.stac.load(
    sentinel2_stac_items,
    bands=["red", "green", "blue", "swir16"],
    chunks={'x': 2048, 'y': 2048},
    bbox=bbox,
    resolution=20,
    groupby="solar_day",
)
sentinel2_dataset

In [ ]:
s2_visible = sentinel2_dataset.max(dim='time')
s2_visible

In [ ]:
config_options = {
                  'AWS_VIRTUAL_HOSTING': 'FALSE',
                  'GDAL_HTTP_UNSAFESSL': 'YES',
                  'AWS_HTTPS': 'YES'}
import rasterio

In [ ]:
worker_env = {
               'AWS_VIRTUAL_HOSTING': 'FALSE',
               'GDAL_HTTP_UNSAFESSL': 'YES',
                  'AWS_HTTPS': 'YES'}

## Dask

In [ ]:
cluster.close()

In [ ]:
from dask_kubernetes.operator import KubeCluster

config = {
   "name": "injhub",
   "namespace": "jhub",
   "image": "guillaumeeb/pangeo-ml-notebook:2026.09.14",
   "n_workers": 2,
   "resources":{"requests": {"memory": "4Gi"}, "limits": {"memory": "4Gi"}}
}

cluster = KubeCluster(**config)

In [ ]:
cluster.scale(20)
cluster

In [ ]:
dask_client = cluster.get_client()
dask_client

In [ ]:
def export_env(env_dict):
    import os
    for key, value in env_dict.items():
        os.environ[key] = value

dask_client.run(export_env, worker_env)

In [ ]:
from distributed.diagnostics.plugin import WorkerPlugin

def configure_workers_environment():
        
    rio_env = rasterio.Env(
        AWS_VIRTUAL_HOSTING='FALSE',
        GDAL_HTTP_UNSAFESSL='YES',
        GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
    )
    rio_env.__enter__()
    return None
    
dask_client.register_worker_callbacks(configure_workers_environment)

### Visualize raw data in RGB

In [ ]:
%%time
with rasterio.Env(**config_options):
    rgb = sentinel2_dataset[["red", "green", "blue"]].to_array(dim="band").max("time")
    rgb_scaled = (rgb / 3000).clip(0, 1)  # Scale and clip for display
    rgb_scaled

In [ ]:
#Visualize subset
with rasterio.Env(**config_options):
    rgb_scaled[:,2000:3000,10000:12000].plot.imshow()

### Calculate NDSI

In [ ]:
%%time
with rasterio.Env(**config_options):
    ndsi = (
        (
            (s2_visible.green - s2_visible.swir16)
            / (s2_visible.green + s2_visible.swir16)
        )
        .where(lambda ndsi: ndsi < 1)
#        .compute()
    )
    
    ndsi[2000:3000,10000:12000].plot.imshow()

## Test out `jupyter-tiler`

...with the `ndvi` `DataArray` we just calculated!

In [ ]:
type(ndsi)

In [ ]:
from jupyter_tiler.titiler import add_data_array

url = await add_data_array(ndsi, colormap_range=(0, 1))

In [ ]:
url

In [ ]:
import leafmap

#center = [41.321482, -72.932739]
center = [42.59, 1.42]
m = leafmap.Map(center=center, zoom=10)
m

In [ ]:
m.add_tile_layer(
    url=url,
    name="NH NDVI",
    attribution="Sentinel 2",
)

In [ ]:
ndsi

In [ ]:
snow = ndsi > 0.4
snow = snow.astype('uint8')
snow

In [ ]:
url_snow = await add_data_array(snow, colormap_range=(0, 1))

In [ ]:
m.add_tile_layer(
    url=url_snow,
    name="snow",
    attribution="Sentinel 2",
)

In [ ]:
cluster.close()